# Proyecto 2 FYHBD | Persona 3 — Capa Gold (`gold_agg.ipynb`)

Ejecución **manual** de respaldo (rúbrica: "cada script debe poder ejecutarse manualmente en caso de fallo en Airflow").
Hace lo mismo que la tarea Gold del DAG: lee las tablas **Silver** de Iceberg (`nessie.silver.users_hist`, `nessie.silver.post_hist`),
aplica reglas de negocio y agregaciones, y escribe las tablas **Gold** (`nessie.gold.*`) con **MERGE** (no override).

* Si Bronze no está cargado, ejecute antes `bronze_ingest.ipynb` (Persona 1).
* Si Silver no existe, ejecute antes `silver_transform.ipynb` (Persona 2).

## 1. Parámetros

In [ ]:
TABLAS = "all"   # all | user_engagement,post_counts_by_user,top_tags,post_activity_monthly
# (el DAG de Airflow ejecuta lo mismo: la tabla del pipeline es user_engagement)

## 2. Importar el módulo de Gold y abrir la sesión Spark (modo local)

Reutiliza el bloque de configuración de Silver (`get_spark`): catálogo Iceberg REST de Nessie + S3 en MinIO.

In [ ]:
import os, sys

# Forzar modo local: ignora cualquier configuración de cluster del contenedor
os.environ["SPARK_MASTER_URL"] = "local[2]"
sys.path.insert(0, "/workspace/scripts")

from gold_agg import run_gold, TABLES
from silver_transform import get_spark

spark = get_spark(master="local[2]", app_name="gold_notebook")
spark.sparkContext.setLogLevel("WARN")
print("Spark OK →", spark.conf.get("spark.sql.catalog.nessie.uri"))

## 3. Agregaciones Gold con MERGE (equivale a la tarea Gold de Airflow)

In [ ]:
tablas = list(TABLES) if TABLAS == "all" else [t.strip() for t in TABLAS.split(",")]
resumenes = run_gold(tablas, spark=spark)   # crea nessie.gold.* si no existen y hace MERGE

## 4. Verificación en Iceberg (consultas vía Spark)

In [ ]:
for r in resumenes:
    fq = r["tabla"]
    print("==", fq)
    spark.sql(f"SELECT * FROM {fq} LIMIT 5").show(truncate=False)
    spark.sql(f"SELECT COUNT(*) filas, MAX(fecha_cargue) ultimo_cargue FROM {fq}").show()

## 5. Idempotencia del MERGE (evidencia)

Re-ejecutar **no duplica** filas: `filas_nuevas = 0` cuando nada cambió (y `claves_duplicadas = 0` siempre).

In [ ]:
rep = run_gold(tablas, spark=spark)
for r in rep:
    print(r["tabla"], "→ filas_nuevas =", r["filas_nuevas"], "| duplicados =", r["claves_duplicadas"])

In [ ]:
spark.stop()